# مرحله‌ی ۱ — Embed + Export (فقط روی Colab)

این نوت‌بوک **هیچ اتصالی به Postgres نداره** — فقط متن‌ها رو با GPU امبد می‌کنه و خروجی سبک (`id` + `embedding`) رو به‌صورت فایل‌های parquet روی Drive ذخیره می‌کنه. کار با Postgres محلی‌تون کاملاً جدا و بعداً، روی خود سیستم‌تون (با اسکریپت دوم) انجام می‌شه — همونی که از اول قرار بود.

### چرا این‌جوری؟
Colab روی سرور گوگل اجراست، نه روی لپ‌تاپ شما — پس اصلاً نمی‌تونه به `localhost` سیستم شما وصل بشه. با جدا کردن کار به دو مرحله، این مشکل کلاً از بین می‌ره: اینجا فقط GPU رایگان Colab رو برای امبدینگ استفاده می‌کنیم، و اتصال به Postgres (که کار ساده‌ایه چون local هست) رو به اسکریپت دوم — که روی خود سیستم شما اجرا می‌شه — می‌سپاریم.

### خروجی
برای هر فایل ورودی، یک فایل خروجی با همون اسم در پوشه‌ی `EMBED_OUT_DIR` ساخته می‌شه، شامل فقط دو ستون: `id`, `embedding`. اگه فایلی رو قبلاً پردازش کرده باشید، دوباره پردازش نمی‌شه (resume خودکار بر اساس وجود فایل خروجی).

#### نصب پکیج‌ها

In [2]:
!pip install -q -U sentence-transformers tqdm pyarrow


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.3/611.3 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 19.2 MB/s eta 0:00:00


#### Imports

In [9]:
from google.colab import drive
drive.flush_and_unmount()
drive.mount("/content/drive", force_remount=True)

import gc
import math
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from sentence_transformers import SentenceTransformer


Mounted at /content/drive


#### Config

In [19]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# ---------- مدل امبدینگ ----------
EMBEDDING_MODEL_NAME = "intfloat/multilingual-e5-base"
USE_E5_PREFIXES = True
EMBED_BATCH_SIZE = 1024      # اگه CUDA OOM گرفتید، بیارید 256
MAX_SEQ_LEN = 128

# ---------- مسیرهای ورودی (روی Drive) ----------
BASE_DIR = Path("/content/drive/MyDrive/Colab Notebooks/Cleaned_data")
COMMENTS_DIR = BASE_DIR / "absa_results_comments"
PRODUCTS_DIR = BASE_DIR / "digikala-products_parts"   # TODO: اسم پوشه رو چک کنید

# ---------- خروجی (روی Drive - فقط id+embedding، خیلی سبک‌تر از دیتای خام) ----------
EMBED_OUT_DIR = BASE_DIR / "embeddings_export"
(EMBED_OUT_DIR / "comments").mkdir(parents=True, exist_ok=True)
(EMBED_OUT_DIR / "products").mkdir(parents=True, exist_ok=True)

# ---------- ستون‌های لازم ----------
COMMENT_ID_COL = "id"
COMMENT_TEXT_COL = "raw_text_normalized"

PRODUCT_ID_COL = "id"
PRODUCT_TEXT_COLS = ["title_fa", "Category1", "Category2", "Brand", "sub_category"]


device: cuda


#### Load Embedding Model

In [20]:
model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=device)
model.max_seq_length = MAX_SEQ_LEN
EMBED_DIM = model.get_embedding_dimension()
print("embedding dim:", EMBED_DIM)


def embed_texts(texts, is_query=False):
    if USE_E5_PREFIXES:
        prefix = "query: " if is_query else "passage: "
        texts = [prefix + (t or "") for t in texts]

    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")):
        vecs = model.encode(
            texts,
            batch_size=EMBED_BATCH_SIZE,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )
    return vecs.astype(np.float16)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

embedding dim: 768


#### Export کامنت‌ها

برای هر فایل ورودی، اگه فایل خروجی هم‌نام از قبل روی Drive باشه، رد می‌شه (resume خودکار).

In [ ]:
def export_comments():
    files = sorted(COMMENTS_DIR.glob("*.parquet"))
    print(f"Found {len(files)} comment files")

    for file in files:
        out_path = EMBED_OUT_DIR / "comments" / file.name
        if out_path.exists():
            print(f"[comments] skip (already exported): {file.name}")
            continue

        df = pd.read_parquet(file, columns=[COMMENT_ID_COL, COMMENT_TEXT_COL])
        n = len(df)
        n_batches = math.ceil(n / EMBED_BATCH_SIZE)

        all_ids = []
        all_embeddings = []

        for b in tqdm(range(n_batches), desc=f"comments:{file.name}"):
            chunk = df.iloc[b * EMBED_BATCH_SIZE : (b + 1) * EMBED_BATCH_SIZE]
            ids = chunk[COMMENT_ID_COL].astype(str).tolist()
            texts = chunk[COMMENT_TEXT_COL].fillna("").astype(str).tolist()

            embeddings = embed_texts(texts, is_query=False)
            all_ids.extend(ids)
            all_embeddings.extend(embeddings.tolist())

        out_df = pd.DataFrame({"id": all_ids, "embedding": all_embeddings})
        out_df.to_parquet(out_path, index=False, engine="pyarrow")
        print(f"Saved -> {out_path}  ({len(out_df)} rows)")

        del df, out_df, all_ids, all_embeddings
        gc.collect()
        if device == "cuda":
            torch.cuda.empty_cache()

    print("\n✅ Comments export finished.")


export_comments()


Found 124 comment files


comments:absa_part_0000.parquet:   0%|          | 0/49 [00:00<?, ?it/s]

Saved -> /content/drive/MyDrive/Colab Notebooks/Cleaned_data/embeddings_export/comments/absa_part_0000.parquet  (50000 rows)


comments:absa_part_0001.parquet:   0%|          | 0/49 [00:00<?, ?it/s]

Saved -> /content/drive/MyDrive/Colab Notebooks/Cleaned_data/embeddings_export/comments/absa_part_0001.parquet  (50000 rows)


...

comments:absa_part_0109.parquet:   0%|          | 0/49 [00:00<?, ?it/s]

Saved -> /content/drive/MyDrive/Colab Notebooks/Cleaned_data/embeddings_export/comments/absa_part_0109.parquet  (50000 rows)


comments:absa_part_0110.parquet:   0%|          | 0/49 [00:00<?, ?it/s]

Saved -> /content/drive/MyDrive/Colab Notebooks/Cleaned_data/embeddings_export/comments/absa_part_0110.parquet  (50000 rows)


comments:absa_part_0111.parquet:   0%|          | 0/49 [00:00<?, ?it/s]